In [ ]:
import os
import random
import sys
from argparse import Namespace
from pathlib import Path
import numpy as np
import torch

# Run from the PatchTST project root so dataset/checkpoint paths resolve correctly.
PATCHTST_ROOT = (Path.cwd().parent / "models" / "patchtst").resolve()
os.chdir(PATCHTST_ROOT)
sys.path.insert(0, str(PATCHTST_ROOT))

In [ ]:
from exp.exp_main import Exp_Main

# User-specified arguments; everything else uses run_longExp.py defaults.
args = Namespace(
    # defaults from scripts/
    random_seed=42,
    is_training=1,
    root_path="../../datasets/",
    data_path="electricity.csv",
    model_id="Electricity_96_96",
    model="PatchTST",
    data="custom",
    features="M",
    seq_len=96,
    pred_len=96,
    enc_in=321,
    e_layers=3,
    n_heads=16,
    d_model=128,
    d_ff=256,
    dropout=0.2,
    fc_dropout=0.2,
    head_dropout=0.0,
    patch_len=16,
    stride=8,
    des="Exp",
    train_epochs=5,
    patience=10,
    lradj="TST",
    pct_start=0.2,
    itr=1,
    batch_size=32,
    learning_rate=0.0005,
    # defaults from run_longExp.py
    target="OT",
    freq="h",
    checkpoints="./checkpoints/",
    label_len=48,
    padding_patch="end",
    revin=1,
    affine=0,
    subtract_last=0,
    decomposition=0,
    kernel_size=25,
    individual=0,
    embed_type=0,
    dec_in=7,
    c_out=7,
    d_layers=1,
    moving_avg=25,
    factor=1,
    distil=True,
    embed="timeF",
    activation="gelu",
    output_attention=False,
    do_predict=False,
    num_workers=10,
    loss="mse",
    use_amp=False,
    use_gpu=True,
    gpu=0,
    use_multi_gpu=False,
    devices="0,1,2,3",
    test_flop=False,
)

# Random seed
random.seed(args.random_seed)
torch.manual_seed(args.random_seed)
np.random.seed(args.random_seed)

args.use_gpu = True if torch.cuda.is_available() and args.use_gpu else False

if args.use_gpu and args.use_multi_gpu:
    args.devices = args.devices.replace(" ", "")
    device_ids = args.devices.split(",")
    args.device_ids = [int(id_) for id_ in device_ids]
    args.gpu = args.device_ids[0]

print("Args in experiment:")
print(args)

Exp = Exp_Main

if args.is_training:
    for ii in range(args.itr):
        setting = "{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}".format(
            args.model_id,
            args.model,
            args.data,
            args.features,
            args.seq_len,
            args.label_len,
            args.pred_len,
            args.d_model,
            args.n_heads,
            args.e_layers,
            args.d_layers,
            args.d_ff,
            args.factor,
            args.embed,
            args.distil,
            args.des,
            ii,
        )

        exp = Exp(args)
        print(">>>>>>>start training : {}>>>>>>>>>>>>>>>>>>>>>>>>>>".format(setting))
        exp.train(setting)

        print(">>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<".format(setting))
        exp.test(setting)

        if args.do_predict:
            print(">>>>>>>predicting : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<".format(setting))
            exp.predict(setting, True)

        torch.cuda.empty_cache()
else:
    ii = 0
    setting = "{}_{}_{}_ft{}_sl{}_ll{}_pl{}_dm{}_nh{}_el{}_dl{}_df{}_fc{}_eb{}_dt{}_{}_{}".format(
        args.model_id,
        args.model,
        args.data,
        args.features,
        args.seq_len,
        args.label_len,
        args.pred_len,
        args.d_model,
        args.n_heads,
        args.e_layers,
        args.d_layers,
        args.d_ff,
        args.factor,
        args.embed,
        args.distil,
        args.des,
        ii,
    )

    exp = Exp(args)
    print(">>>>>>>testing : {}<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<<".format(setting))
    exp.test(setting, test=1)
    torch.cuda.empty_cache()

Args in experiment:
Namespace(random_seed=42, is_training=1, root_path='../../datasets/', data_path='electricity.csv', model_id='Electricity_96_96', model='PatchTST', data='custom', features='M', seq_len=96, pred_len=96, enc_in=321, e_layers=3, n_heads=16, d_model=128, d_ff=256, dropout=0.2, fc_dropout=0.2, head_dropout=0.0, patch_len=16, stride=8, des='Exp', train_epochs=5, patience=10, lradj='TST', pct_start=0.2, itr=1, batch_size=32, learning_rate=0.0005, target='OT', freq='h', checkpoints='./checkpoints/', label_len=48, padding_patch='end', revin=1, affine=0, subtract_last=0, decomposition=0, kernel_size=25, individual=0, embed_type=0, dec_in=7, c_out=7, d_layers=1, moving_avg=25, factor=1, distil=True, embed='timeF', activation='gelu', output_attention=False, do_predict=False, num_workers=10, loss='mse', use_amp=False, use_gpu=True, gpu=0, use_multi_gpu=False, devices='0,1,2,3', test_flop=False)
Use GPU: cuda:0
>>>>>>>start training : Electricity_96_96_PatchTST_custom_ftM_sl96_ll4